# Topic Modeling
### Bert Sentence Embedding, LDA, and K-means Clustering

#### Concatenate the sentence embeddings with LDA, use an autoencoder to get the latent representation and perform K-means to perform clustering on the data

#### After the sentences have been assigned to their clusters, we can extract the candidate topics for each cluster using tokens of it's sentences

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')


In [ ]:
% cd /content/drive/My Drive/Thesis_colab

In [ ]:
! pip install transformers
! pip install pandas
! pip install torch
! pip install nltk
! pip install gensim
! pip install matplotlib
! pip install umap-learn
! pip install wordcloud
! pip install sklearn
! pip install tqdm
! pip install stop_words
! pip install xlrd
! pip install langdetect
! pip install openpyxl

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

### 1. Reading and preprocessing the data

In [ ]:
from utilities.document_preprocessing import preprocess_documents

#### The preprocess_document function reads the sentences from the file and generates tokens from each sentences.

In [ ]:
# Input file
file = "data/PMP_Skills_Matching_Pilot_2018_2019_small.xlsx"

In [1]:
Verbose = False

In [2]:

#read the previously processed data
import pandas as pd
data = pd.read_pickle("output/preprocessed_data/use_this_data.pkl")
sentences = list(data['sentences'])
token_lists = list(data['token_list'])


In [3]:
token_lists[0]

['leadership', 'strategi', 'support', 'project', 'initi']

In [ ]:
sentences, token_lists = preprocess_documents(file, verbose = Verbose)

### 2. Topic Modeling

In [4]:
from models.topic_model import Topic_Model

#### The topic_Model class does the follow
- Extract Bert sentence embeddings for all sentences (Local features)
- get topic distribution for each sentence using LDA
- concatenate both and learn the latent representation using an auto-encoder
- cluster the latent representation into K clusters

In [5]:
# Define the topic model object
TopicModelObject = Topic_Model(k = 2, verbose = Verbose, embedding = 'bert', saved_embedding_dir = None, embedding_batch_size = 100)

In [ ]:
# Fit the topic model
TopicModelObject.fit(sentences, token_lists)

Getting vector representations for LDA 
Getting vector representations for LDA. Done! 


 Getting vector representations for bert 
cpu
Embedding for batch 0 out of 2 batches Completed.


## Results

In [ ]:
from utilities.visualizations import *
%matplotlib inline

### Visualizations for LDA only

In [ ]:
lda_class_distribution = TopicModelObject.vec["LDA"]
cluster_label_per_document = np.amax(lda_class_distribution, axis=1)
print (cluster_label_per_document.shape)

In [ ]:
# To get the top words for each topic, this method only uses LDA, doesn't utilize Bert or K mean clustering
#TopicModelObject.get_top_words_lda(top=30)

In [ ]:
for k , cluster_word_distribution in TopicModelObject.get_top_words_lda(top=100).items():
    get_wordcloud(TopicModelObject, word_distribution = cluster_word_distribution, topic = k )

### Visualizations for LDA + Embeddings
#### Only run when embeddings have been used

In [ ]:
labels = set(TopicModelObject.cluster_model.labels_)
for label in labels:
    index = []
    for idx, l in enumerate(TopicModelObject.cluster_model.labels_):
        if l == label:
            index.append(idx)
    print ('\n \nSentences belong to cluster ', str (label+1), ' are: \n ')
    for i in index:
        #print (sentences[i])
        continue

#### Topic Modeling for 1-gram

In [ ]:
# visualize and save img 
#visualize(TopicModelObject)
for i in range(TopicModelObject.k):
    get_wordcloud(TopicModelObject, token_lists, i)

#### Topic Modeling for 2-gram

In [ ]:
from utilities.ngram import generate_n_gram

In [ ]:
for i in range(TopicModelObject.k):
    ngram_tokens_list = generate_n_gram(sentences, n=2, separator = '-' ,
                                        keep_stopwords= False, stem=False, keep_nouns_only=False)
    
    ngram_token_count = dict()
    for sentence in ngram_tokens_list:
        for feature in sentence:
            if feature in ngram_token_count.keys():
                ngram_token_count[feature] += 1
            else:
                ngram_token_count[feature] = 1    
    
    get_wordcloud(TopicModelObject, topic=i, word_count_dict = ngram_token_count)

#### Topic Modeling for 3-gram

In [ ]:
from utilities.ngram import generate_n_gram

In [ ]:
for i in range(TopicModelObject.k):
    ngram_tokens_list = generate_n_gram(sentences, n=3, separator = '-' ,
                                        keep_stopwords= False, stem=False, keep_nouns_only=False)
    
    ngram_token_count = dict()
    for sentence in ngram_tokens_list:
        for feature in sentence:
            if feature in ngram_token_count.keys():
                ngram_token_count[feature] += 1
            else:
                ngram_token_count[feature] = 1    
    
    get_wordcloud(TopicModelObject, topic=i, word_count_dict = ngram_token_count)